In [0]:
# CREATE SILVER DATABASE
 
spark.sql("CREATE DATABASE IF NOT EXISTS tijartek_silver")
print(" tijartek_silver database ready")

 tijartek_silver database ready


In [0]:
# IMPORTS
 
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import date
 
print(" Imports done")

 Imports done


In [0]:
# SILVER - LOCATION

spark.sql("CREATE SCHEMA IF NOT EXISTS tijartek_silver")
 
location = spark.table("tijartek_bronze.locations")
 
silver_location = location \
    .withColumn("location_id", F.col("location_id").cast(IntegerType())) \
    .withColumn("city",        F.trim(F.col("city"))) \
    .withColumn("region",      F.trim(F.col("region"))) \
    .withColumn("country",     F.lit("Egypt")) \
    .withColumn("timezone",    F.lit("Africa/Cairo")) \
    .drop("_source", "_ingested_at", "_source_table")
 
silver_location.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_silver.location")
 
print(f" location : {silver_location.count():,} rows")
 

 location : 50 rows


In [0]:
#  SILVER - CATEGORY
# Clean: trim name, validate commission rate

from pyspark.sql.types import IntegerType, DecimalType, StringType

category = spark.table("tijartek_bronze.categories")

silver_category = category \
    .withColumn("category_id",     F.col("category_id").cast(IntegerType())) \
    .withColumn("name",            F.trim(F.col("name"))) \
    .withColumn("commission_rate", F.col("commission_rate").cast(DecimalType(5, 2))) \
    .withColumn("commission_rate", F.when(F.col("commission_rate").isNull(), F.lit(0.0))
                                    .otherwise(F.col("commission_rate"))) \
    .withColumn("parent_category", F.lit(None).cast(StringType())) \
    .drop("_source", "_ingested_at", "_source_table")

silver_category.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_silver.category")

print(f" category: {silver_category.count():,} rows")

 category: 25 rows


In [0]:
#  SILVER - SELLER
# Clean: trim name, standardize type, add seller_tier
 
seller = spark.table("tijartek_bronze.sellers")
 
silver_seller = seller \
    .withColumn("seller_id",   F.col("seller_id").cast(IntegerType())) \
    .withColumn("name",        F.trim(F.col("name"))) \
    .withColumn("type",        F.trim(F.col("type"))) \
    .withColumn("type",        F.when(F.col("type").isNull(), F.lit("Unknown"))
                                .otherwise(F.col("type"))) \
    .withColumn("join_date",   F.col("join_date").cast(DateType())) \
    .withColumn("location_id", F.col("location_id").cast(IntegerType())) \
    .withColumn("seller_tier", F.lit("Standard")) \
    .drop("_source", "_ingested_at", "_source_table")
 
silver_seller.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_silver.seller")
 
print(f" seller: {silver_seller.count():,} rows")

 seller: 1,000 rows


In [0]:
# SILVER - CUSTOMER
# Clean: email lowercase, phone standardize, add age + age_band
# -----------------------------------------------------------------------------
 
customer = spark.table("tijartek_bronze.customers")
 
silver_customer = customer \
    .withColumn("customer_id",        F.col("customer_id").cast(IntegerType())) \
    .withColumn("name",               F.trim(F.col("name"))) \
    .withColumn("email",              F.lower(F.trim(F.col("email")))) \
    .withColumn("phone",              F.trim(F.col("phone"))) \
    .withColumn("dob",                F.col("dob").cast(DateType())) \
    .withColumn("gender",             F.initcap(F.trim(F.col("gender")))) \
    .withColumn("gender",             F.when(~F.col("gender").isin("Male", "Female", "Other"), F.lit("Unknown"))
                                       .otherwise(F.col("gender"))) \
    .withColumn("registration_date",  F.col("registration_date").cast(DateType())) \
    .withColumn("location_id",        F.col("location_id").cast(IntegerType())) \
    .withColumn("age",                F.floor(F.datediff(F.current_date(), F.col("dob")) / 365)) \
    .withColumn("age_band",
        F.when(F.col("age") < 25,  F.lit("18-24"))
         .when(F.col("age") < 35,  F.lit("25-34"))
         .when(F.col("age") < 45,  F.lit("35-44"))
         .when(F.col("age") < 55,  F.lit("45-54"))
         .when(F.col("age") < 65,  F.lit("55-64"))
         .otherwise(F.lit("65+"))) \
    .drop("_source", "_ingested_at", "_source_table")
 
silver_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_silver.customer")
 
print(f" customer: {silver_customer.count():,} rows")

 customer: 80,000 rows


In [0]:
#  SILVER - PRODUCT
# Clean: price/stock nulls, standardize is_fragile (True/False → 1/0)

 
product = spark.table("tijartek_bronze.products")
 
silver_product = product \
    .withColumn("product_id",     F.col("product_id").cast(IntegerType())) \
    .withColumn("seller_id",      F.col("seller_id").cast(IntegerType())) \
    .withColumn("category_id",    F.col("category_id").cast(IntegerType())) \
    .withColumn("name",           F.trim(F.col("name"))) \
    .withColumn("description",    F.trim(F.col("description"))) \
    .withColumn("price",          F.col("price").cast(DecimalType(10, 2))) \
    .withColumn("price",          F.when(F.col("price").isNull() | (F.col("price") <= 0), F.lit(0.01))
                                   .otherwise(F.col("price"))) \
    .withColumn("stock_quantity", F.col("stock_quantity").cast(IntegerType())) \
    .withColumn("stock_quantity", F.when(F.col("stock_quantity").isNull(), F.lit(0))
                                   .otherwise(F.col("stock_quantity"))) \
    .withColumn("weight",         F.col("weight").cast(DecimalType(8, 2))) \
    .withColumn("is_fragile",     F.when(F.col("is_fragile").cast(StringType()).isin("True", "true", "1"), F.lit(1))
                                   .otherwise(F.lit(0))) \
    .drop("_source", "_ingested_at", "_source_table")
 
silver_product.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_silver.product")
 
print(f" product: {silver_product.count():,} rows")

 product: 50,000 rows


In [0]:
#  SILVER - PROMOTIONS
# Clean: dates, discount_type standardize

promotions = spark.table("tijartek_bronze.promotions")
 
silver_promotions = promotions \
    .withColumn("promotion_id",  F.col("promotion_id").cast(IntegerType())) \
    .withColumn("name",          F.trim(F.col("name"))) \
    .withColumn("discount_type", F.lower(F.trim(F.col("discount_type")))) \
    .withColumn("start_date",    F.col("start_date").cast(DateType())) \
    .withColumn("end_date",      F.col("end_date").cast(DateType())) \
    .withColumn("budget",        F.col("budget").cast(DecimalType(12, 2))) \
    .withColumn("platform",      F.trim(F.col("platform"))) \
    .withColumn("duration_days", F.datediff(F.col("end_date"), F.col("start_date"))) \
    .drop("_source", "_ingested_at", "_source_table")
 
silver_promotions.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_silver.promotions")
 
print(f" promotions: {silver_promotions.count():,} rows")

 promotions: 500 rows


In [0]:
#  SILVER - SESSION
# Clean: timestamps, calculate session duration

 
session = spark.table("tijartek_bronze.sessions")
 
silver_session = session \
    .withColumn("session_id",       F.col("session_id").cast(LongType())) \
    .withColumn("customer_id",      F.col("customer_id").cast(IntegerType())) \
    .withColumn("device_type",      F.lower(F.trim(F.col("device_type")))) \
    .withColumn("start_timestamp",  F.col("start_timestamp").cast(TimestampType())) \
    .withColumn("end_timestamp",    F.col("end_timestamp").cast(TimestampType())) \
    .withColumn("duration_minutes",
        F.round((F.unix_timestamp("end_timestamp") - F.unix_timestamp("start_timestamp")) / 60, 2)) \
    .withColumn("duration_minutes",
        F.when(F.col("duration_minutes") < 0, F.lit(None))
         .otherwise(F.col("duration_minutes"))) \
    .drop("_source", "_ingested_at", "_source_table")
 
silver_session.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_silver.session")
 
print(f" session: {silver_session.count():,} rows")
 

 session: 120,000 rows


In [0]:
#  SILVER - USER_EVENT
# Clean: standardize event_type and page_type

 
user_event = spark.table("tijartek_bronze.user_events")
 
silver_user_event = user_event \
    .withColumn("event_id",        F.col("event_id").cast(LongType())) \
    .withColumn("session_id",      F.col("session_id").cast(LongType())) \
    .withColumn("product_id",      F.col("product_id").cast(IntegerType())) \
    .withColumn("event_type",      F.lower(F.trim(F.col("event_type")))) \
    .withColumn("page_type",       F.lower(F.trim(F.col("page_type")))) \
    .withColumn("event_timestamp", F.col("event_timestamp").cast(TimestampType())) \
    .withColumn("event_date",      F.to_date(F.col("event_timestamp"))) \
    .withColumn("event_hour",      F.hour(F.col("event_timestamp"))) \
    .drop("_source", "_ingested_at", "_source_table")
 
silver_user_event.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_silver.user_event")
 
print(f" user_event: {silver_user_event.count():,} rows")

 user_event: 500,000 rows


In [0]:
# SILVER - ORDER
# Clean: status standardize, nulls in promotion_id is OK
 
order = spark.table("tijartek_bronze.orders")
 
silver_order = order \
    .withColumn("order_id",      F.col("order_id").cast(LongType())) \
    .withColumn("customer_id",   F.col("customer_id").cast(IntegerType())) \
    .withColumn("promotion_id",  F.col("promotion_id").cast(IntegerType())) \
    .withColumn("date",          F.col("date").cast(TimestampType())) \
    .withColumn("order_date",    F.to_date(F.col("date"))) \
    .withColumn("order_hour",    F.hour(F.col("date"))) \
    .withColumn("status",        F.lower(F.trim(F.col("status")))) \
    .withColumn("total_amount",  F.col("total_amount").cast(DecimalType(12, 2))) \
    .withColumn("discount",      F.col("discount").cast(DecimalType(10, 2))) \
    .withColumn("discount",      F.when(F.col("discount").isNull(), F.lit(0.0))
                                  .otherwise(F.col("discount"))) \
    .withColumn("shipping_fee",  F.col("shipping_fee").cast(DecimalType(10, 2))) \
    .withColumn("shipping_fee",  F.when(F.col("shipping_fee").isNull(), F.lit(0.0))
                                  .otherwise(F.col("shipping_fee"))) \
    .withColumn("has_promotion", F.when(F.col("promotion_id").isNull(), F.lit(0))
                                  .otherwise(F.lit(1))) \
    .drop("_source", "_ingested_at", "_source_table")
 
silver_order.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_silver.order")
 
print(f" order: {silver_order.count():,} rows")
 

 order: 150,000 rows


In [0]:
# SILVER - ORDER_ITEM
# Clean: calculate line total
 
order_item = spark.table("tijartek_bronze.order_items")
 
silver_order_item = order_item \
    .withColumn("order_item_id", F.col("order_item_id").cast(LongType())) \
    .withColumn("order_id",      F.col("order_id").cast(LongType())) \
    .withColumn("product_id",    F.col("product_id").cast(IntegerType())) \
    .withColumn("quantity",      F.col("quantity").cast(IntegerType())) \
    .withColumn("unit_price",    F.col("unit_price").cast(DecimalType(10, 2))) \
    .withColumn("tax_amount",    F.col("tax_amount").cast(DecimalType(10, 2))) \
    .withColumn("tax_amount",    F.when(F.col("tax_amount").isNull(), F.lit(0.0))
                                  .otherwise(F.col("tax_amount"))) \
    .withColumn("line_total",    F.round(F.col("quantity") * F.col("unit_price"), 2)) \
    .withColumn("gross_revenue", F.round(F.col("line_total") + F.col("tax_amount"), 2)) \
    .drop("_source", "_ingested_at", "_source_table")
 
silver_order_item.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_silver.order_item")
 
print(f" order_item: {silver_order_item.count():,} rows")
 

 order_item: 338,051 rows


In [0]:
#  SILVER - PAYMENT
# Clean: standardize method and status
 
payment = spark.table("tijartek_bronze.payments")
 
silver_payment = payment \
    .withColumn("payment_id",   F.col("payment_id").cast(LongType())) \
    .withColumn("order_id",     F.col("order_id").cast(LongType())) \
    .withColumn("method",       F.initcap(F.trim(F.col("method")))) \
    .withColumn("status",       F.lower(F.trim(F.col("status")))) \
    .withColumn("amount",       F.col("amount").cast(DecimalType(12, 2))) \
    .withColumn("payment_date", F.col("payment_date").cast(TimestampType())) \
    .withColumn("is_cod",       F.when(F.upper(F.col("method")) == "COD", F.lit(1))
                                 .otherwise(F.lit(0))) \
    .drop("_source", "_ingested_at", "_source_table")
 
silver_payment.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_silver.payment")
 
print(f" payment: {silver_payment.count():,} rows")
 

 payment: 150,000 rows


In [0]:
# SILVER - SHIPMENT
# Clean: calculate delivery_days, is_on_time
 
shipment = spark.table("tijartek_bronze.shipments")
 
silver_shipment = shipment \
    .withColumn("shipment_id",        F.col("shipment_id").cast(LongType())) \
    .withColumn("order_id",           F.col("order_id").cast(LongType())) \
    .withColumn("status",             F.lower(F.trim(F.col("status")))) \
    .withColumn("shipped_date",       F.col("shipped_date").cast(DateType())) \
    .withColumn("delivered_date",     F.col("delivered_date").cast(DateType())) \
    .withColumn("estimated_arrival",  F.col("estimated_arrival").cast(DateType())) \
    .withColumn("location_id",        F.col("location_id").cast(IntegerType())) \
    .withColumn("delivery_days",
        F.datediff(F.col("delivered_date"), F.col("shipped_date"))) \
    .withColumn("is_on_time",
        F.when(F.col("delivered_date") <= F.col("estimated_arrival"), F.lit(1))
         .when(F.col("delivered_date").isNull(), F.lit(None))
         .otherwise(F.lit(0))) \
    .drop("_source", "_ingested_at", "_source_table")
 
silver_shipment.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_silver.shipment")
 
print(f" shipment: {silver_shipment.count():,} rows")

 shipment: 131,981 rows


In [0]:
# SILVER - RETURN
# Clean: standardize status, fix nulls
 
returns = spark.table("tijartek_bronze.returns")
 
silver_return = returns \
    .withColumn("return_id",      F.col("return_id").cast(LongType())) \
    .withColumn("order_item_id",  F.col("order_item_id").cast(LongType())) \
    .withColumn("status",         F.lower(F.trim(F.col("status")))) \
    .withColumn("reason",         F.trim(F.col("reason"))) \
    .withColumn("reason",         F.when(F.col("reason").isNull(), F.lit("Not specified"))
                                   .otherwise(F.col("reason"))) \
    .withColumn("refund_amount",  F.col("refund_amount").cast(DecimalType(10, 2))) \
    .withColumn("refund_amount",  F.when(F.col("refund_amount").isNull(), F.lit(0.0))
                                   .otherwise(F.col("refund_amount"))) \
    .withColumn("date",           F.col("date").cast(DateType())) \
    .drop("_source", "_ingested_at", "_source_table")
 
silver_return.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_silver.return")
 
print(f" return: {silver_return.count():,} rows")

 return: 26,726 rows


In [0]:
#  SILVER - REVIEW
# Clean: validate rating range, add sentiment bucket
 
review = spark.table("tijartek_bronze.reviews")
 
silver_review = review \
    .withColumn("review_id",   F.col("review_id").cast(LongType())) \
    .withColumn("customer_id", F.col("customer_id").cast(IntegerType())) \
    .withColumn("product_id",  F.col("product_id").cast(IntegerType())) \
    .withColumn("rating",      F.col("rating").cast(DecimalType(2, 1))) \
    .withColumn("rating",      F.when(F.col("rating") > 5, F.lit(5.0))
                                .when(F.col("rating") < 1, F.lit(1.0))
                                .otherwise(F.col("rating"))) \
    .withColumn("text",        F.trim(F.col("text"))) \
    .withColumn("date",        F.col("date").cast(DateType())) \
    .withColumn("sentiment",
        F.when(F.col("rating") >= 4.0, F.lit("Positive"))
         .when(F.col("rating") >= 3.0, F.lit("Neutral"))
         .otherwise(F.lit("Negative"))) \
    .drop("_source", "_ingested_at", "_source_table")
 
silver_review.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_silver.review")
 
print(f" review: {silver_review.count():,} rows")
 

 review: 60,000 rows


In [0]:
# SILVER - INVENTORY_LOG
# Clean: standardize transaction_type
 
inventory_log = spark.table("tijartek_bronze.inventory_logs")
 
silver_inventory_log = inventory_log \
    .withColumn("log_id",           F.col("log_id").cast(LongType())) \
    .withColumn("product_id",       F.col("product_id").cast(IntegerType())) \
    .withColumn("log_date",         F.col("log_date").cast(TimestampType())) \
    .withColumn("log_date_only",    F.to_date(F.col("log_date"))) \
    .withColumn("quantity_change",  F.col("quantity_change").cast(IntegerType())) \
    .withColumn("transaction_type", F.lower(F.trim(F.col("transaction_type")))) \
    .withColumn("stock_direction",
        F.when(F.col("quantity_change") > 0, F.lit("IN"))
         .otherwise(F.lit("OUT"))) \
    .drop("_source", "_ingested_at", "_source_table")
 
silver_inventory_log.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_silver.inventory_log")
 
print(f" inventory_log: {silver_inventory_log.count():,} rows")

 inventory_log: 100,000 rows


In [0]:
# VERIFY SILVER LAYER

 
print("\n" + "=" * 50)
print("SILVER LAYER - ROW COUNTS")
print("=" * 50)
 
silver_tables = [
    "location", "category", "seller", "customer", "product",
    "promotions", "session", "user_event", "order", "order_item",
    "payment", "shipment", "return", "review", "inventory_log"
]
 
total = 0
for t in silver_tables:
    count = spark.table(f"tijartek_silver.{t}").count()
    total += count
    print(f"{t:20} {count:>10,} rows")
 
print("-" * 40)
print(f"{'TOTAL':20} {total:>10,} rows")
print("=" * 50)
print("\n SILVER LAYER COMPLETE!")


SILVER LAYER - ROW COUNTS
location                     50 rows
category                     25 rows
seller                    1,000 rows
customer                 80,000 rows
product                  50,000 rows
promotions                  500 rows
session                 120,000 rows
user_event              500,000 rows
order                   150,000 rows
order_item              338,051 rows
payment                 150,000 rows
shipment                131,981 rows
return                   26,726 rows
review                   60,000 rows
inventory_log           100,000 rows
----------------------------------------
TOTAL                 1,708,333 rows

 SILVER LAYER COMPLETE!
